## Creates support tables

Tables created in this script
- patterns_meta
- verb_phrase_matches

In [1]:
import sqlite3
import pandas as pd
import sys
sys.path.append("..")
from common_display import display_db_table 

### Configuration

In [2]:
DB_DIR = "../example_data"

PATTERN_DB = f"{DB_DIR}/verb_patterns.db"
TRANSACTION_DB = f"{DB_DIR}/transactions.db"

PATTERNS_TABLE = "patterns"
VERB_PHRASE_MATCHES_TABLE = "verb_phrase_matches"
PATTERNS_META_TABLE = "patterns_meta"

## Connect to db

In [3]:
con = sqlite3.connect(PATTERN_DB)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" as trans ')

## Workflow

### Abitabel asjade kättesaamiseks

Mitte-elegantne viis saada kätte kõik **head_id**-d, millele vastavates fraasides on esindatud kõik vaadeldavate mustrite osised (sobiv kääne (kui on), kaassõna (kui on), infiniitverb (kui on)). Saab kasutada ülejäänud tabelite koostamiseks. Ilmselt on võimalik teha tegelikult ära ka JOIN-ide abil. 

In [4]:
%%time

cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step1""")

cur.execute("""
CREATE TABLE verb_phrase_matches_step1 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    phrase_nr
FROM 
(
    SELECT 
        pat.pat_id as pat_id,
        tr_head.id as head_id,
        pat.phrase_case as phrase_case,
        pat.phrase_nr as phrase_nr
    FROM 
        {tbl} as pat
    INNER JOIN 
        trans.transaction_head as tr_head
    ON
        pat.verb_word=tr_head.verb
    WHERE
        pat.verb_compound=tr_head.verb_compound
) as pat_tr_joined
INNER JOIN 
    trans.transaction_row as tr
ON
    pat_tr_joined.head_id=tr.head_id
""".format(tbl=PATTERNS_TABLE))

CPU times: user 26 ms, sys: 975 µs, total: 27 ms
Wall time: 29.5 ms


In [5]:
display_db_table(con, "verb_phrase_matches_step1")

,pat_id,head_id,phrase_case,phrase_nr
0,1,54,abl,1
1,1,54,abl,1
2,1,74,abl,1
3,1,74,abl,1
4,1,74,abl,1


In [6]:
%%time

cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step2""")

cur.execute("""
CREATE TABLE verb_phrase_matches_step2 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    phrase_nr
FROM 
    verb_phrase_matches_step1 as step1
INNER JOIN 
    trans.transaction_row as tr
ON 
    step1.head_id=tr.head_id
""")

CPU times: user 23.9 ms, sys: 4.4 ms, total: 28.3 ms
Wall time: 31.9 ms


In [7]:
display_db_table(con, "verb_phrase_matches_step2")

,pat_id,head_id,phrase_case,phrase_nr
0,1,54,abl,1
1,1,54,abl,1
2,1,54,abl,1
3,1,54,abl,1
4,1,74,abl,1


In [8]:
%%time

cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step3""")

cur.execute("""
CREATE TABLE verb_phrase_matches_step3 AS
SELECT 
    pat_id,
    tr.head_id,
    phrase_case,
    phrase_nr
FROM 
    verb_phrase_matches_step2 as step2
INNER JOIN
    trans.transaction_row as tr
ON 
    step2.head_id=tr.head_id
""")

CPU times: user 71.4 ms, sys: 4.57 ms, total: 76 ms
Wall time: 83.5 ms


In [9]:
display_db_table(con, "verb_phrase_matches_step3")

,pat_id,head_id,phrase_case,phrase_nr
0,1,54,abl,1
1,1,54,abl,1
2,1,54,abl,1
3,1,54,abl,1
4,1,54,abl,1


In [10]:
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step1""")
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step2""")

### I tabel patterns_meta

In [11]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=PATTERNS_META_TABLE))

cur.execute("""
CREATE TABLE {tbl} AS 
SELECT 
    pat_id,
    count(*) AS phrase_count
FROM
(
    SELECT DISTINCT
        pat_id, 
        head_id
    FROM
        verb_phrase_matches_step3
) AS tbl
GROUP BY
    tbl.pat_id
ORDER BY
    phrase_count DESC
""".format(tbl=PATTERNS_META_TABLE))

cur.execute("""CREATE INDEX meta_pat_id_idx ON {tbl}(pat_id)""".format(tbl=PATTERNS_META_TABLE))

con.commit()

CPU times: user 29.7 ms, sys: 2.85 ms, total: 32.5 ms
Wall time: 37.3 ms


In [12]:
display_db_table(con, PATTERNS_META_TABLE)

,pat_id,phrase_count
0,9611,20
1,8611,20
2,7609,20
3,4592,20
4,2585,20


### II tabel verb_phrase_matches

In [13]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""
CREATE TABLE {tbl} AS
SELECT DISTINCT
    pat_id,
    head_id,
    phrase_nr
FROM
    verb_phrase_matches_step3
""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

cur.execute("""CREATE INDEX match_pat_id_idx ON {tbl}(pat_id)""".format(tbl=VERB_PHRASE_MATCHES_TABLE))
cur.execute("""CREATE INDEX match_head_id_idx ON {tbl}(head_id)""".format(tbl=VERB_PHRASE_MATCHES_TABLE))
cur.execute("""CREATE INDEX match_phrase_nr_idx ON {tbl}(phrase_nr)""".format(tbl=VERB_PHRASE_MATCHES_TABLE))

con.commit()

CPU times: user 32.7 ms, sys: 5.41 ms, total: 38.1 ms
Wall time: 50.6 ms


In [14]:
display_db_table(con, VERB_PHRASE_MATCHES_TABLE)

,pat_id,head_id,phrase_nr
0,1,54,1
1,1,74,1
2,1,96,1
3,1,130,1
4,1,206,1


In [ ]:
cur.execute("""DROP TABLE IF EXISTS verb_phrase_matches_step3""")

In [15]:
con.close()